# 06 · Xception feature extractor + classical ensemble
**Final code set · 2026-10-22 · individual notebook**

**What this notebook does.** Frozen Xception at 128 x 128 with a 256-value Dense layer (`dense_3`) used as the feature vector, then Random Forest, Decision Tree and SVM with soft voting.

**Run first:** nothing, this notebook is independent.  
**Approximate time on a Colab T4 GPU:** 10-20 min.

**Saves (on Google Drive, in `brain_tumor_results_2026-10-22/`):**
- `models/xception_ext.h5`
- `models/xception_feat_rf.pkl, _dt.pkl, _svm.pkl`
- `history/xception_ext_history.csv`
- metrics + confusion files for xception_feat_*

**How to run:** `Runtime > Change runtime type > T4 GPU`, then run every cell from top to bottom. If Colab disconnects, reconnect and run all cells again: training resumes from the last finished epoch and finished work is skipped.

Cells 1 to 7 are the same in every notebook (Drive, settings, logging, data loading). The cells after them are specific to this model.

In [ ]:
# 1. Mount Google Drive (results are saved here so they survive disconnects)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration  --  EDIT THE TWO PATHS BELOW IF NEEDED
import os

DRIVE_DATA_DIR = '/content/drive/MyDrive/MRI_DATASET'                # your dataset on Drive (Training/ + Testing/)
OUTPUT_DIR     = '/content/drive/MyDrive/brain_tumor_results_2026-10-22'  # all results are written here

# Hugging Face: trained models are uploaded here and the web app downloads them from here
HF_REPO_ID     = 'shehank98/brain-tumor-mri-models'
RESULTS_PREFIX = 'results_2026-10-22'

# The final ensemble refuses to run unless all 6 predictors exist (keeps the report honest)
ALLOW_PARTIAL_ENSEMBLE = False

# Explainability method for the EAA-IoU analysis: 'hirescam' (default, correct for every model) or 'gradcam'
CAM_METHOD = 'hirescam'

# Reading thousands of images from Drive every epoch is very slow and is a common reason
# long Colab runs stall. The dataset is copied ONCE to Colab's fast local disk and read from there.
import shutil
LOCAL_DATA_DIR = '/content/MRI_DATASET'
if not os.path.isdir(os.path.join(LOCAL_DATA_DIR, 'Testing')):
    print('Copying dataset from Drive to local disk (one time, about 1 to 3 minutes) ...')
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)
DATA_DIR = LOCAL_DATA_DIR

# Hyperparameters (kept identical to the project so results are comparable)
EPOCHS_CNN       = 20
EPOCHS_TRANSFER  = 10
EPOCHS_EXTRACTOR = 10
BATCH    = 32
SIZE_STD = 256      # standalone CNN / InceptionV3 / Xception
SIZE_EXT = 128      # feature extractors that feed classical ML
SEED     = 42
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
N_CLASSES   = 4

MODELS_DIR  = os.path.join(OUTPUT_DIR, 'models')
METRICS_DIR = os.path.join(OUTPUT_DIR, 'metrics')
HISTORY_DIR = os.path.join(OUTPUT_DIR, 'history')
CONF_DIR    = os.path.join(OUTPUT_DIR, 'confusion')
LOG_DIR     = os.path.join(OUTPUT_DIR, 'logs')
for d in [MODELS_DIR, METRICS_DIR, HISTORY_DIR, CONF_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
MASTER_CSV  = os.path.join(METRICS_DIR, 'results_master.csv')
RUN_LOG     = os.path.join(LOG_DIR, 'run_log.txt')
PROGRESS    = os.path.join(OUTPUT_DIR, 'progress.json')

# sanity check: every class folder must exist and contain images; the real counts are saved for the report
import json
counts = {}
for split in ['Training', 'Testing']:
    counts[split] = {}
    for c in CLASS_NAMES:
        pth = os.path.join(DATA_DIR, split, c)
        n = len(os.listdir(pth)) if os.path.isdir(pth) else 0
        counts[split][c] = n
        print('{:8s} {:11s} {:5d} images {}'.format(split, c, n, '' if n else '  <-- MISSING, check DRIVE_DATA_DIR'))
    print('{:8s} {:11s} {:5d} images in total'.format(split, '', sum(counts[split].values())))
json.dump(counts, open(os.path.join(METRICS_DIR, 'dataset_counts.json'), 'w'), indent=2)
print('Results ->', OUTPUT_DIR)

In [ ]:
# 3. Imports, seeds, logging and progress tracking
import json, time, csv, random, datetime, traceback
import numpy as np
random.seed(SEED); np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.callbacks import Callback, CSVLogger, ModelCheckpoint
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import joblib

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE  (Runtime > Change runtime type > GPU)')

def log(msg):
    line = '[{:%Y-%m-%d %H:%M:%S}] {}'.format(datetime.datetime.now(), msg)
    print(line)
    with open(RUN_LOG, 'a') as f:
        f.write(line + '\n')

def _progress():
    return json.load(open(PROGRESS)) if os.path.exists(PROGRESS) else {}

def mark(stage, status, info=''):
    data = _progress()
    data[stage] = {'status': status, 'info': str(info)[:300], 'time': str(datetime.datetime.now())}
    json.dump(data, open(PROGRESS, 'w'), indent=2)

def is_done(stage):
    return _progress().get(stage, {}).get('status') == 'done'

def stage(name, fn):
    # Runs fn() guarded: logs any error and keeps going so one failure never loses the rest.
    if is_done(name):
        log('SKIP ' + name + ' (already done)'); return
    mark(name, 'running')
    try:
        fn(); mark(name, 'done'); log('DONE ' + name)
    except Exception as e:
        mark(name, 'error', e)
        log('ERROR in ' + name + ': ' + repr(e) + '\n' + traceback.format_exc())
        print('\n>>> ' + name + ' failed but was isolated. Fix and re-run THIS cell; other stages are safe.')

In [ ]:
# 4. Metric logging  --  saves accuracy and every other number as DATA (csv/json), not images
def save_confusion(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    with open(os.path.join(CONF_DIR, name + '_confusion_matrix.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['actual_vs_predicted'] + CLASS_NAMES)
        for i, row in enumerate(cm):
            w.writerow([CLASS_NAMES[i]] + [int(v) for v in row])
    return cm

def log_metrics(name, y_true, y_pred, train_time=None, note=''):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _  = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(N_CLASSES)), zero_division=0)
    rep = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                zero_division=0, output_dict=True)
    json.dump(rep, open(os.path.join(METRICS_DIR, name + '_classification_report.json'), 'w'), indent=2)
    save_confusion(name, y_true, y_pred)
    header = (['model','accuracy','precision_macro','recall_macro','f1_macro']
              + ['f1_' + c for c in CLASS_NAMES]
              + ['n_test','train_time_sec','timestamp','note'])
    row = ([name, round(acc,4), round(p,4), round(r,4), round(f,4)]
           + [round(float(x),4) for x in fc]
           + [int(len(y_true)), round(float(train_time or 0),1),
              str(datetime.datetime.now()), note])
    new = not os.path.exists(MASTER_CSV)
    with open(MASTER_CSV, 'a', newline='') as fcsv:
        w = csv.writer(fcsv)
        if new: w.writerow(header)
        w.writerow(row)
    log('{}: accuracy={:.2f}%   macro-F1={:.3f}'.format(name, acc*100, f))
    return acc

In [ ]:
# 5. Resumable training  --  checkpoints every epoch; a crash resumes, it does not restart
class EpochTracker(Callback):
    def __init__(self, path): super().__init__(); self.path = path
    def on_epoch_end(self, epoch, logs=None):
        json.dump({'epoch': int(epoch) + 1}, open(self.path, 'w'))

def resumable_fit(name, build_fn, train_ds, val_ds, epochs):
    final_path = os.path.join(MODELS_DIR, name + '.h5')
    ckpt_path  = os.path.join(MODELS_DIR, name + '_ckpt.keras')
    epoch_path = os.path.join(MODELS_DIR, name + '_epoch.json')
    if os.path.exists(final_path):
        log(name + ': final model already exists, loading')
        return load_model(final_path)
    init_epoch = 0
    if os.path.exists(ckpt_path) and os.path.exists(epoch_path):
        model = load_model(ckpt_path)
        init_epoch = json.load(open(epoch_path)).get('epoch', 0)
        log('{}: resuming from epoch {}'.format(name, init_epoch))
    else:
        model = build_fn()
    cbs = [ModelCheckpoint(ckpt_path, save_freq='epoch', save_best_only=False),
           CSVLogger(os.path.join(HISTORY_DIR, name + '_history.csv'), append=True),
           EpochTracker(epoch_path)]
    if init_epoch < epochs:
        model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                  initial_epoch=init_epoch, callbacks=cbs, verbose=1)
    model.save(final_path)                       # permanent copy on Drive
    for p in (ckpt_path, epoch_path):
        try: os.remove(p)
        except OSError: pass
    return model

In [ ]:
# 6. Data loaders
def tf_datasets(size):
    common = dict(image_size=(size, size), batch_size=BATCH, label_mode='int', seed=SEED)
    tr = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'Training'), validation_split=0.1, subset='training',
        shuffle=True, **common)
    va = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'Training'), validation_split=0.1, subset='validation',
        shuffle=True, **common)
    te = tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, 'Testing'), shuffle=False,
        image_size=(size, size), batch_size=BATCH, label_mode='int')
    norm = lambda x, y: (x / 255.0, y)
    AT = tf.data.AUTOTUNE
    return tr.map(norm).prefetch(AT), va.map(norm).prefetch(AT), te.map(norm).prefetch(AT)

def load_numpy(split, size, grayscale=False):
    import cv2
    X, y = [], []
    base = os.path.join(DATA_DIR, split)
    flag = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR
    for idx, c in enumerate(CLASS_NAMES):
        folder = os.path.join(base, c)
        for fn in sorted(os.listdir(folder)):
            img = cv2.imread(os.path.join(folder, fn), flag)
            if img is None: continue
            img = cv2.resize(img, (size, size))
            X.append(img); y.append(idx)
    X = np.asarray(X, dtype='float32') / 255.0
    if not grayscale:
        X = X[..., ::-1]           # BGR(cv2) -> RGB to match the Keras loaders
    return X, np.asarray(y)

In [ ]:
# 7. Shared prediction, feature-extraction and classical-ML helpers (same in every notebook)
def predict_ds(model, ds):
    yt, yp = [], []
    for X, y in ds:
        p = model.predict(X, verbose=0)
        yp.extend(np.argmax(p, axis=1)); yt.extend(y.numpy())
    return np.asarray(yt), np.asarray(yp)

def extract_features(model, layer_name, X, batch=64):
    try:
        ex = Model(model.inputs, model.get_layer(layer_name).output)
        return ex.predict(X, batch_size=batch, verbose=1)
    except (AttributeError, ValueError):
        # Keras 3 Sequential without a defined input: run the layers by hand up to the feature layer
        out = []
        for s in range(0, len(X), batch):
            h = tf.convert_to_tensor(X[s:s + batch], tf.float32)
            for layer in model.layers:
                h = layer(h, training=False)
                if layer.name == layer_name:
                    break
            out.append(h.numpy())
        return np.concatenate(out)

def train_classical(tag, Xtr, ytr, Xte, yte, include_lr=False):
    # Each classifier is trained and SAVED separately, so a crash in one keeps the others.
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.svm import SVC
    from sklearn.linear_model import LogisticRegression
    specs = {'rf':  RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
             'dt':  DecisionTreeClassifier(random_state=SEED),
             'svm': SVC(probability=True, random_state=SEED, cache_size=1000)}
    if include_lr:
        specs['lr'] = LogisticRegression(max_iter=1000, n_jobs=-1)
    fitted = {}
    for key, clf in specs.items():
        mp = os.path.join(MODELS_DIR, tag + '_' + key + '.pkl')
        if os.path.exists(mp):
            fitted[key] = joblib.load(mp); log(tag + '_' + key + ': loaded (skip)'); continue
        t = time.time(); log(tag + '_' + key + ': training ...')
        clf.fit(Xtr, ytr); joblib.dump(clf, mp)
        log_metrics(tag + '_' + key, yte, clf.predict(Xte), time.time() - t)
        fitted[key] = clf
    vote_keys = [k for k in ['rf','dt','svm'] if k in fitted]
    probs = sum(fitted[k].predict_proba(Xte) for k in vote_keys) / len(vote_keys)
    log_metrics(tag + '_softvote_ensemble', yte, np.argmax(probs, axis=1))
    return fitted

## This notebook: Xception feature extractor + classical ensemble

In [ ]:
# 8. Model definition
def _base(kind, size):
    App = tf.keras.applications.InceptionV3 if kind == 'inception' else tf.keras.applications.Xception
    b = App(input_shape=(size, size, 3), weights='imagenet', include_top=False)
    b.trainable = False
    return b

def build_transfer_extractor(kind, feat_name, size=128):
    base = _base(kind, size)
    x = layers.Flatten()(base.output)
    x = layers.Dense(256, activation='relu', name=feat_name)(x)
    x = layers.Dropout(0.5, name='feature_dropout')(x)
    out = layers.Dense(N_CLASSES, activation='softmax', name='predictions')(x)
    m = Model(base.input, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

In [ ]:
# 9. Training, evaluation and logging
def _run_feature_ensemble(ext_name, build_ext, layer, tag):
    tr, va, _ = tf_datasets(SIZE_EXT)
    ext = resumable_fit(ext_name, build_ext, tr, va, EPOCHS_EXTRACTOR)
    Xtr, ytr = load_numpy('Training', SIZE_EXT)
    Xte, yte = load_numpy('Testing',  SIZE_EXT)
    ftr = extract_features(ext, layer, Xtr)
    fte = extract_features(ext, layer, Xte)
    del Xtr, Xte
    train_classical(tag, ftr, ytr, fte, yte)

def run_xception_ensemble():
    _run_feature_ensemble('xception_ext',
        lambda: build_transfer_extractor('xception', 'dense_3'), 'dense_3', 'xception_feat')

In [ ]:
# Run it (safe to re-run: resumes or skips finished work)
STAGE = 'Xception_feature_ensemble'
stage(STAGE, run_xception_ensemble)

In [ ]:
# Results for this notebook (read back from Google Drive)
if not is_done(STAGE):
    raise SystemExit("This stage has not finished yet. Re-run the cell above; it resumes where it stopped.")
import pandas as pd
df = pd.read_csv(MASTER_CSV).drop_duplicates("model", keep="last")
mine = df[df.model.str.startswith(('xception_feat_',))]
print(mine[["model","accuracy","precision_macro","recall_macro","f1_macro","train_time_sec"]].to_string(index=False))
print("\nConfusion matrix (xception_feat_softvote_ensemble):")
print(pd.read_csv(os.path.join(CONF_DIR, "xception_feat_softvote_ensemble_confusion_matrix.csv")).to_string(index=False))
print("\nTraining history (xception_ext), last epochs:")
print(pd.read_csv(os.path.join(HISTORY_DIR, "xception_ext_history.csv")).tail().to_string(index=False))